# Statistical repair (Phase R1 -- R-2, R-4, R-7)

Corrects the multiplicity problem the peer review flagged: the paper's one
"significant" result (ablation stage1-vs-stage3, McNemar p=0.032) was reported
without correcting for the fact that it is one test among several implied by the
paper's own tables. This notebook:

1. Refits the four ablation stages (train.csv only, exactly reproducing
   `ablation_v1.ipynb`'s definitions) plus Naive Bayes text-only and
   Naive Bayes text+metadata (train+valid pool, matching Table II's reported best
   model, macro-F1 0.655) so the *actual reported best model* is included in the
   significance family, not just the LR ablation stages (fixes R-7).
2. Persists every model's test-set predictions to `test_predictions_v2.csv` so
   every paired test in the paper is reproducible from one file.
3. Runs the full McNemar family and corrects with Holm-Bonferroni
   (`statsmodels.stats.multitest.multipletests`, method="holm") -- uniformly more
   powerful than plain Bonferroni, same family-wise error control.
4. Adds paired bootstrap 95% CIs on macro-F1 *differences* (not just per-model
   point estimates) for the headline comparisons, plus McNemar discordant-pair
   counts and their exact binomial CI.
5. Cross-seed test variance for every model in the main results table.

**On the Holm correction:** the acceptance plan predicts this step kills the
p=0.032 headline (5-8 comparisons -> p_adj ~ 0.16-0.25). That is treated as the
correct outcome here, not something to route around -- see
`claude-workspace/CLAUDE_ACCEPTANCE_PLAN.md` Section 6, contingency C1.

**Note on the joint family with Phase R2:** the Holm correction applied in this
notebook covers only the R1 comparisons below. `generalization_audit_v1.ipynb`
(R2) adds its own Wilcoxon tests to the *same* family rather than running a
second, separately-corrected family -- see that notebook's final cell for the
combined, authoritative adjusted p-values used in the manuscript.

In [1]:
import re

import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from scipy import stats
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.proportion import proportion_confint

from sklearn.feature_selection import SelectKBest, chi2
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

from liar_utils import RANDOM_STATE, evaluate_full, load_and_label
from metadata_features import METADATA_COLUMNS, _build_canonical_categories, _fillna_str, build_pipeline_transformer

SEEDS = [42, 1, 7, 13, 2024]

Load data. Ablation stages (S1-S4) are refit on `train.csv` only, exactly matching `ablation_v1.ipynb`'s definitions, so their test predictions here are identical to that notebook's. The Naive Bayes models are fit on `train.csv` + `valid.csv` (the pool every other v2+ notebook uses, per `CLAUDE.md`), matching Table II's reported numbers.

In [2]:
train = load_and_label("train.csv")
valid = load_and_label("valid.csv")
test = load_and_label("test.csv")
train_full = pd.concat([train, valid], ignore_index=True)

stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()


def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z]", " ", text)
    words = text.split()
    words = [stemmer.stem(w) for w in words if w not in stop_words]
    return " ".join(words)


for df in (train, train_full, test):
    df["clean_text"] = df["Statement"].apply(preprocess)

canonical_meta_s = _build_canonical_categories(train, METADATA_COLUMNS)
train = _fillna_str(train, METADATA_COLUMNS, canonical_meta_s)
test_s = _fillna_str(test, METADATA_COLUMNS, canonical_meta_s)

canonical_meta_full = _build_canonical_categories(train_full, METADATA_COLUMNS)
train_full = _fillna_str(train_full, METADATA_COLUMNS, canonical_meta_full)
test_full = _fillna_str(test, METADATA_COLUMNS, canonical_meta_full)

y_train, y_test = train["Label"], test_s["Label"]
y_train_full = train_full["Label"]
assert (test_s["Label"].values == test_full["Label"].values).all()
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

Ablation stages 1-4 (Logistic Regression, `train.csv` only) -- reproduces `ablation_v1.ipynb`'s stage definitions to get the same test predictions.

In [3]:
def make_pipeline(use_metadata, k_features, clf):
    steps = [("features", build_pipeline_transformer(use_metadata=use_metadata))]
    if k_features is not None:
        steps.append(("select", SelectKBest(chi2, k=k_features)))
    steps.append(("clf", clf))
    return Pipeline(steps)


stage_defs = {
    "S1_text_only": dict(use_metadata=False, k_features=None),
    "S2_metadata": dict(use_metadata=True, k_features=None),
    "S3_feature_selection": dict(use_metadata=True, k_features=3000),
}

test_predictions = {"y_true": y_test.to_numpy()}

for name, cfg in stage_defs.items():
    pipe = make_pipeline(cfg["use_metadata"], cfg["k_features"], LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    pipe.fit(train, y_train)
    test_predictions[name] = pipe.predict(test_s)
    print(name, "test macro-F1 =", round(evaluate_full(y_test, test_predictions[name])["macro_f1"], 6))

tuned_pipe = make_pipeline(use_metadata=True, k_features=3000, clf=LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
param_grid = {"clf__C": [0.1, 1, 10], "clf__class_weight": [None, "balanced"]}
grid = GridSearchCV(tuned_pipe, param_grid, cv=cv, scoring="f1_macro", n_jobs=-1)
grid.fit(train, y_train)
test_predictions["S4_tuning"] = grid.best_estimator_.predict(test_s)
print("S4_tuning test macro-F1 =", round(evaluate_full(y_test, test_predictions["S4_tuning"])["macro_f1"], 6))

S1_text_only test macro-F1 = 0.596461


/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


S2_metadata test macro-F1 = 0.598694


S3_feature_selection test macro-F1 = 0.630048


/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


...

/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parame

/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parame

S4_tuning test macro-F1 = 0.621845


/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Naive Bayes text-only and text+metadata, `train+valid` pool, `alpha=0.5` (Table II's selected hyperparameter for the best model). This is also the single "NB text-only" number R5.1 will use to reconcile the baseline notebook's inconsistent TF-IDF configuration.

In [4]:
nb_defs = {
    "NB_text_only": dict(use_metadata=False),
    "NB_text_metadata": dict(use_metadata=True),
}

for name, cfg in nb_defs.items():
    pipe = make_pipeline(cfg["use_metadata"], None, MultinomialNB(alpha=0.5))
    pipe.fit(train_full, y_train_full)
    test_predictions[name] = pipe.predict(test_full)
    print(name, "test macro-F1 =", round(evaluate_full(y_test, test_predictions[name])["macro_f1"], 6))

NB_text_only test macro-F1 = 0.60351


NB_text_metadata test macro-F1 = 0.6549


/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [5]:
pred_df = pd.DataFrame(test_predictions)
pred_df.to_csv("test_predictions_v2.csv", index=False)
pred_df.head()

,y_true,S1_text_only,S2_metadata,S3_feature_selection,S4_tuning,NB_text_only,NB_text_metadata
0,1,1,1,0,0,1,0
1,0,1,1,1,1,1,1
2,0,0,0,0,0,0,0
3,1,1,1,1,1,1,1
4,0,0,0,0,0,0,0


## R1.1/R1.2 -- comparison family and Holm-Bonferroni correction

Every McNemar test the paper makes or implies, in one family:
- the five ablation-stage comparisons (S1-S2, S1-S3, S1-S4, S2-S3, S3-S4)
- the best-model metadata gain (NB text+metadata vs NB text-only)
- the best model vs the previously-reported validated recipe (NB text+metadata vs
  S3, i.e. LR + feature selection)

(DistilBERT is not in this family -- Phase R4, which would fairly retrain it on
the same data pool, was deferred under this revision's minimum-viable scope; see
`CLAUDE.md`/`MEMORY.md`.)

In [6]:
def mcnemar_pair(name_a, name_b, y_true, preds):
    a_correct = preds[name_a] == y_true
    b_correct = preds[name_b] == y_true
    both = int(np.sum(a_correct & b_correct))
    only_a = int(np.sum(a_correct & ~b_correct))
    only_b = int(np.sum(~a_correct & b_correct))
    neither = int(np.sum(~a_correct & ~b_correct))
    table = [[both, only_a], [only_b, neither]]
    result = mcnemar(table, exact=False, correction=True)
    b, c = only_a, only_b
    n_disc = b + c
    if n_disc > 0:
        ci_low, ci_high = proportion_confint(count=b, nobs=n_disc, method="beta")
    else:
        ci_low, ci_high = (np.nan, np.nan)
    return dict(
        comparison=f"{name_a} vs {name_b}",
        statistic=result.statistic,
        p_raw=result.pvalue,
        b_disc=b,
        c_disc=c,
        b_over_bpc=b / n_disc if n_disc > 0 else np.nan,
        b_over_bpc_ci_low=ci_low,
        b_over_bpc_ci_high=ci_high,
    )


y_true_arr = pred_df["y_true"].to_numpy()
comparisons = [
    ("S1_text_only", "S2_metadata"),
    ("S1_text_only", "S3_feature_selection"),
    ("S1_text_only", "S4_tuning"),
    ("S2_metadata", "S3_feature_selection"),
    ("S3_feature_selection", "S4_tuning"),
    ("NB_text_metadata", "NB_text_only"),
    ("NB_text_metadata", "S3_feature_selection"),
]

r1_family = pd.DataFrame([mcnemar_pair(a, b, y_true_arr, test_predictions) for a, b in comparisons])
r1_family.insert(0, "phase", "R1")
r1_family

,phase,comparison,statistic,p_raw,b_disc,c_disc,b_over_bpc,b_over_bpc_ci_low,b_over_bpc_ci_high
0,R1,S1_text_only vs S2_metadata,0.000000,1.000000,124,125,0.497992,0.434229,0.561803
1,R1,S1_text_only vs S3_feature_selection,4.612100,0.031747,122,159,0.434164,0.375399,0.494332
2,R1,S1_text_only vs S4_tuning,0.189911,0.662991,164,173,0.486647,0.432121,0.541409
3,R1,S2_metadata vs S3_feature_selection,9.141791,0.002498,49,85,0.365672,0.284216,0.453206
4,R1,S3_feature_selection vs S4_tuning,4.925676,0.026460,88,60,0.594595,0.510880,0.674443
5,R1,NB_text_metadata vs NB_text_only,8.694340,0.003192,157,108,0.592453,0.530643,0.652171
6,R1,NB_text_metadata vs S3_feature_selection,2.116402,0.145729,105,84,0.555556,0.481672,0.627669


In [7]:
reject, p_holm_r1_only, _, _ = multipletests(r1_family["p_raw"], alpha=0.05, method="holm")
r1_family["p_holm_R1_family_only"] = p_holm_r1_only
r1_family["significant_R1_family_only"] = reject
r1_family.to_csv("pvalue_family_r1.csv", index=False)
print("Provisional (R1-comparisons-only) Holm correction -- superseded once R2's Wilcoxon")
print("tests are folded into the same family; see generalization_audit_v1.ipynb's final cell.")
r1_family[["comparison", "p_raw", "p_holm_R1_family_only", "significant_R1_family_only"]]

Provisional (R1-comparisons-only) Holm correction -- superseded once R2's Wilcoxon
tests are folded into the same family; see generalization_audit_v1.ipynb's final cell.


,comparison,p_raw,p_holm_R1_family_only,significant_R1_family_only
0,S1_text_only vs S2_metadata,1.000000,1.000000,False
1,S1_text_only vs S3_feature_selection,0.031747,0.132302,False
2,S1_text_only vs S4_tuning,0.662991,1.000000,False
3,S2_metadata vs S3_feature_selection,0.002498,0.017489,True
4,S3_feature_selection vs S4_tuning,0.026460,0.132302,False
5,NB_text_metadata vs NB_text_only,0.003192,0.019152,True
6,NB_text_metadata vs S3_feature_selection,0.145729,0.437186,False


## R1.3 -- paired bootstrap 95% CI on macro-F1 differences

For each headline comparison: resample test-set indices once per iteration,
score both models on the *same* resample, and take the difference. This is more
informative than two separate per-model CIs -- it directly answers "how big
could the true gain be" for each pairing.

In [8]:
def paired_bootstrap_diff_ci(name_a, name_b, y_true, preds, n_boot=5000, seed=RANDOM_STATE):
    rng = np.random.RandomState(seed)
    n = len(y_true)
    diffs = np.empty(n_boot)
    pa, pb = preds[name_a], preds[name_b]
    for i in range(n_boot):
        idx = rng.randint(0, n, n)
        f1_a = f1_score(y_true[idx], pa[idx], average="macro")
        f1_b = f1_score(y_true[idx], pb[idx], average="macro")
        diffs[i] = f1_a - f1_b
    ci_low, ci_high = np.percentile(diffs, [2.5, 97.5])
    point = f1_score(y_true, pa, average="macro") - f1_score(y_true, pb, average="macro")
    return dict(comparison=f"{name_a} minus {name_b}", point_diff=point, ci_low=ci_low, ci_high=ci_high)


headline_pairs = [
    ("S3_feature_selection", "S1_text_only"),
    ("NB_text_metadata", "NB_text_only"),
    ("NB_text_metadata", "S3_feature_selection"),
]

bootstrap_diffs = pd.DataFrame(
    [paired_bootstrap_diff_ci(a, b, y_true_arr, test_predictions) for a, b in headline_pairs]
)
bootstrap_diffs.to_csv("bootstrap_diff_ci_v1.csv", index=False)
bootstrap_diffs

,comparison,point_diff,ci_low,ci_high
0,S3_feature_selection minus S1_text_only,0.033587,0.006301,0.061053
1,NB_text_metadata minus NB_text_only,0.051390,0.024611,0.078245
2,NB_text_metadata minus S3_feature_selection,0.024852,0.003196,0.047019


## R1.4 -- best model is now in the tested family

`NB_text_metadata` (Table II's actual best, macro-F1 0.655) has its own gain
over `NB_text_only` tested above, on the same footing as the LR ablation. The
cell below states explicitly whether NB's gain and LR's (stage1-vs-stage3) gain
survive the Holm correction the same way or differently -- if they diverge, the
manuscript must say so rather than defaulting to the LR-only narrative.

In [9]:
nb_row = r1_family[r1_family["comparison"] == "NB_text_metadata vs NB_text_only"].iloc[0]
lr_row = r1_family[r1_family["comparison"] == "S1_text_only vs S3_feature_selection"].iloc[0]
print(f"NB text+metadata vs NB text-only: p_raw={nb_row.p_raw:.6f}, p_holm(R1-only)={nb_row.p_holm_R1_family_only:.6f}, significant={nb_row.significant_R1_family_only}")
print(f"LR stage1 vs stage3 (previous headline): p_raw={lr_row.p_raw:.6f}, p_holm(R1-only)={lr_row.p_holm_R1_family_only:.6f}, significant={lr_row.significant_R1_family_only}")

NB text+metadata vs NB text-only: p_raw=0.003192, p_holm(R1-only)=0.019152, significant=True
LR stage1 vs stage3 (previous headline): p_raw=0.031747, p_holm(R1-only)=0.132302, significant=False


## R1.5 -- cross-seed test variance

For every model reported in the current Table II headline row for its pipeline
family ("Text + metadata", no speaker -- the config Table II reports as each
model's best), refit across seeds `{42, 1, 7, 13, 2024}` and report test
macro-F1 mean +/- std. LR/SVM/NB are fit with deterministic solvers given a fixed
split, so their std is expected to be exactly 0.000 -- that is reported as a
result, not treated as a gap. RandomForest/XGBoost are stochastic estimators and
are expected to show non-zero std.

In [10]:
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

def model_for_seed(name, seed):
    if name == "NB":
        return MultinomialNB(alpha=0.5)
    if name == "LR":
        return LogisticRegression(C=1, class_weight="balanced", solver="liblinear", max_iter=1000, random_state=seed)
    if name == "SVM":
        return LinearSVC(C=0.1, class_weight="balanced", random_state=seed)
    if name == "RF":
        return RandomForestClassifier(max_depth=None, min_samples_split=2, n_estimators=200, random_state=seed, n_jobs=-1)
    if name == "XGBoost":
        return XGBClassifier(learning_rate=0.1, max_depth=6, n_estimators=200, random_state=seed, eval_metric="logloss")
    raise ValueError(name)


seed_rows = []
for model_name in ["NB", "LR", "SVM", "RF", "XGBoost"]:
    scores = []
    for seed in SEEDS:
        pipe = make_pipeline(True, None, model_for_seed(model_name, seed))
        pipe.fit(train_full, y_train_full)
        y_pred = pipe.predict(test_full)
        scores.append(f1_score(y_test, y_pred, average="macro"))
    scores = np.array(scores)
    seed_rows.append(dict(model=model_name, config="text+metadata", mean_macro_f1=scores.mean(), std_macro_f1=scores.std(), seeds=str(SEEDS), scores=str(list(np.round(scores, 6)))))
    print(model_name, "mean=", round(scores.mean(), 6), "std=", round(scores.std(), 6))

seed_variance = pd.DataFrame(seed_rows)
seed_variance.to_csv("seed_variance_v1.csv", index=False)
seed_variance

/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


...

/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


XGBoost mean= 0.617406 std= 0.0


,model,config,mean_macro_f1,std_macro_f1,seeds,scores
0,NB,text+metadata,0.654900,0.000000,"[42, 1, 7, 13, 2024]","[np.float64(0.6549), np.float64(0.6549), np.fl..."
1,LR,text+metadata,0.622884,0.000000,"[42, 1, 7, 13, 2024]","[np.float64(0.622884), np.float64(0.622884), n..."
2,SVM,text+metadata,0.621998,0.000000,"[42, 1, 7, 13, 2024]","[np.float64(0.621998), np.float64(0.621998), n..."
3,RF,text+metadata,0.622256,0.003312,"[42, 1, 7, 13, 2024]","[np.float64(0.625566), np.float64(0.621795), n..."
4,XGBoost,text+metadata,0.617406,0.000000,"[42, 1, 7, 13, 2024]","[np.float64(0.617406), np.float64(0.617406), n..."


## Summary

- **The Holm correction does what the acceptance plan predicted, but only for
  the LR ablation's headline.** The previously-reported "significant" result
  (stage1-vs-stage3, LR, p_raw=0.0317) does **not** survive: p_holm=0.132 across
  the 7-comparison R1 family. It must no longer be described as statistically
  significant.
- **It does not kill the paper's actual best-model result.** Naive Bayes
  text+metadata vs. Naive Bayes text-only (the Table II best, macro-F1 0.655 vs.
  0.604) is significant both raw (p=0.0032) and Holm-adjusted (p=0.0192) even in
  this provisional, R1-only family -- it will only get more conservative once
  R2's tests are folded in, but a raw p this small has room to survive that too.
  Paired bootstrap 95% CI on the difference: [0.025, 0.078] macro-F1, clearly
  excluding zero. **R1.4's fork resolves in the paper's favor**: the model
  actually reported as best in Table II has a metadata gain that is significant
  under correction; the LR ablation's gain, examined in isolation, is not. Report
  both, explicitly, rather than defaulting to whichever one is more convenient.
- The intermediate step S2-vs-S3 (adding feature selection on top of metadata)
  is itself significant (p_holm=0.017), which is a more precise localization of
  where the LR ablation's real effect lives than the original S1-vs-S3 framing.
- NB_text_metadata vs. S3_feature_selection (best model vs. the previously
  "validated recipe") is not significant (p_holm=0.44) -- consistent with both
  being legitimate, similarly-performing configurations rather than one clearly
  beating the other.
- Cross-seed variance: LR, SVM, and NB are exactly deterministic on this fixed
  data/config (std=0.000 across 5 seeds) -- expected and worth stating plainly,
  not a gap. XGBoost was also exactly deterministic here (std=0.000) given fixed
  hyperparameters and `eval_metric="logloss"`. RandomForest is the only model
  with non-trivial cross-seed spread (mean 0.6223, std 0.0033) -- small relative
  to the inter-model gaps in Table II, so it does not change any ranking claim.
- **Do not treat these Holm-adjusted p-values as final for the manuscript.**
  `generalization_audit_v1.ipynb` (R2) adds its own Wilcoxon tests to this same
  family and re-runs the correction jointly at the end of that notebook -- *that*
  combined table, not `pvalue_family_r1.csv`, is what R6 should cite.
